---
# **Flow 5 — Deploy Gradio**

**Prinsip deploy:** notebook ini dan aplikasi Gradio **terpisah**. Cell di bawah hanya
meng-export artefak (bobot model, konfigurasi, koordinat graf, skenario contoh, hasil
benchmarking) ke folder `gradio_artifacts/`. Folder ini kemudian dipindah ke direktori
yang sama dengan `app.py` dan dijalankan lokal / di-deploy ke railway Spaces / server
sendiri — bukan dijalankan langsung dari dalam Colab.


In [ ]:
# ===== FLOW 5 (a) — Export Artefak untuk Gradio Demo =====
# Cell ini TIDAK mengubah variabel/model manapun dari cell sebelumnya.
# Ia hanya MEMBACA hasil yang sudah ada (G, model, lstm_model, edge_index, speed_matrix,
# config, test_df/test_norm, sample_nodes, comparison_full/sample, horizon_maes, dst)
# lalu membungkusnya jadi artefak siap-pakai untuk app.py (Gradio), yang dijalankan TERPISAH.

import os, pickle, json
import numpy as np
import pandas as pd
import torch
from torchdiffeq import odeint

ART_DIR = 'gradio_artifacts'
os.makedirs(ART_DIR, exist_ok=True)

# ---------- 1. Koordinat node asli (OSM lat/lon) untuk visualisasi peta jaringan ----------
node_lat = np.array([G.nodes[n]['y'] for n in node_list], dtype=np.float32)
node_lon = np.array([G.nodes[n]['x'] for n in node_list], dtype=np.float32)

# ---------- 2. Edge list (index) untuk menggambar ruas jalan ----------
edge_list_idx = [(node_id_to_idx[u], node_id_to_idx[v]) for u, v, k in G.edges(keys=True)]

# ---------- 3. Skenario contoh (tombol "Examples") ----------
# Ambil window seq_len (1 jam historis) berakhir tepat di titik waktu representatif.
speed_matrix_novel = speed_matrix.drop(columns=[c for c in ['day'] if c in speed_matrix.columns])

def get_scenario_window(center_ts_str):
    center_ts = pd.Timestamp(center_ts_str)
    pos = speed_matrix_novel.index.get_indexer([center_ts], method='nearest')[0]
    window = speed_matrix_novel.iloc[pos - seq_len + 1: pos + 1]
    assert len(window) == seq_len, f"Window tidak lengkap utk {center_ts_str}, cek rentang data"
    return window.values.astype(np.float32)  # [seq_len, num_nodes], satuan km/h asli

scenario_defs = {
    'Jam Sibuk Pagi (Senin 08:00)':  '2026-01-06 08:00',
    'Jam Sibuk Sore (Senin 17:30)':  '2026-01-06 17:30',
    'Malam Lengang (Senin 21:00)':   '2026-01-06 21:00',
    'Weekend Siang (Sabtu 14:00)':   '2026-01-11 14:00',
}
examples = {label: get_scenario_window(ts) for label, ts in scenario_defs.items()}

print("Skenario contoh berhasil diambil:")
for label, arr in examples.items():
    print(f"  - {label}: shape {arr.shape}, rata-rata {arr.mean():.1f} km/h")

# ---------- 4. Error vs Horizon -- lengkapi LSTM & ARIMA agar sebanding dgn GNN-ODE ----------
# GNN-ODE sudah punya evaluasi multi-horizon di CELL 38 (variabel horizon_maes, HORIZONS).
# Blok di bawah MELENGKAPI dengan LSTM (rollout autoregresif) & ARIMA (forecast multi-step
# native), dibatasi pada horizon & 20 node sample YANG SAMA dgn benchmarking sebelumnya --
# supaya perbandingan tetap apples-to-apples dan konsisten dgn disclaimer yang sudah ada.

HORIZONS_CMP = [1, 2, 3]  # x5 menit -> 5, 10, 15 menit
max_h = max(HORIZONS_CMP)

# --- 4a. GNN-ODE: reuse dari CELL 38 (horizon_maes), tambahkan RMSE per horizon ---
gnn_ode_mae_h, gnn_ode_rmse_h = [], []
model.eval()
_multi_preds_h = {h: [] for h in HORIZONS_CMP}
_multi_targets_h = {h: [] for h in HORIZONS_CMP}
t_span_multi_export = torch.tensor([0.] + [float(h) for h in HORIZONS_CMP]).to(device)
with torch.no_grad():
    for start in range(0, len(test_norm) - seq_len - max_h, 4):
        x = torch.tensor(test_norm[start:start + seq_len], dtype=torch.float32).unsqueeze(0).to(device)
        h0 = model.encoder(x[:, -1, :].unsqueeze(-1))
        h_traj = odeint(model.odefunc, h0[0], t_span_multi_export, method=model.solver)
        for i, h in enumerate(HORIZONS_CMP):
            pred = model.decoder(h_traj[i + 1]).squeeze(-1).cpu().numpy()
            _multi_preds_h[h].append(pred)
            _multi_targets_h[h].append(test_norm[start + seq_len + h - 1])

for h in HORIZONS_CMP:
    p = np.array(_multi_preds_h[h]) * train_std + train_mean
    t = np.array(_multi_targets_h[h]) * train_std + train_mean
    gnn_ode_mae_h.append(float(np.mean(np.abs(p - t))))
    gnn_ode_rmse_h.append(float(np.sqrt(np.mean((p - t) ** 2))))

# --- 4b. LSTM: rollout autoregresif (prediksi step-t dipakai sbg input step-(t+1)) ---
lstm_mae_h, lstm_rmse_h = [], []
_lstm_preds_h = {h: [] for h in HORIZONS_CMP}
_lstm_targets_h = {h: [] for h in HORIZONS_CMP}
lstm_model.eval()
with torch.no_grad():
    for start in range(0, len(test_norm) - seq_len - max_h, 4):
        x = torch.tensor(test_norm[start:start + seq_len], dtype=torch.float32).unsqueeze(0).to(device)
        cur_seq = x.clone()
        for h in range(1, max_h + 1):
            pred = lstm_model(cur_seq)  # [1, num_nodes]
            if h in HORIZONS_CMP:
                _lstm_preds_h[h].append(pred.cpu().numpy()[0])
                _lstm_targets_h[h].append(test_norm[start + seq_len + h - 1])
            cur_seq = torch.cat([cur_seq[:, 1:, :], pred.unsqueeze(1)], dim=1)

for h in HORIZONS_CMP:
    p = np.array(_lstm_preds_h[h]) * train_std + train_mean
    t = np.array(_lstm_targets_h[h]) * train_std + train_mean
    lstm_mae_h.append(float(np.mean(np.abs(p - t))))
    lstm_rmse_h.append(float(np.sqrt(np.mean((p - t) ** 2))))

# --- 4c. ARIMA: forecast(steps=max_h) native multi-step, node sample yg sama (20 node) ---
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

arima_mae_h_acc = {h: [] for h in HORIZONS_CMP}
arima_targets_h_acc = {h: [] for h in HORIZONS_CMP}
ARIMA_STRIDE = 12  # 12 x 5menit = 1 jam antar origin, biar tidak terlalu berat

for node in sample_nodes:
    train_series = train_df[node].values
    test_series_full = test_df[node].values
    try:
        arima_res = ARIMA(train_series, order=(2, 1, 1)).fit()
    except Exception:
        continue
    current_res = arima_res
    for start in range(0, len(test_series_full) - max_h, ARIMA_STRIDE):
        fc = current_res.forecast(steps=max_h)
        for i, h in enumerate(HORIZONS_CMP):
            arima_mae_h_acc[h].append(fc[i] - test_series_full[start + h - 1])
        # update state pakai observasi asli s.d. origin berikutnya (rolling, tanpa refit)
        n_step = min(ARIMA_STRIDE, len(test_series_full) - start)
        current_res = current_res.append(test_series_full[start:start + n_step], refit=False)

arima_mae_h, arima_rmse_h = [], []
for h in HORIZONS_CMP:
    errs = np.array(arima_mae_h_acc[h])
    arima_mae_h.append(float(np.mean(np.abs(errs))))
    arima_rmse_h.append(float(np.sqrt(np.mean(errs ** 2))))

print("\n=== Error vs Horizon (MAE, km/h) -- 20 node sample, konsisten dgn benchmarking ARIMA ===")
horizon_labels_min = [h * 5 for h in HORIZONS_CMP]
print(pd.DataFrame({
    'Horizon (menit)': horizon_labels_min,
    'GNN-ODE': gnn_ode_mae_h,
    'LSTM': lstm_mae_h,
    'ARIMA': arima_mae_h,
}).to_string(index=False))

# ---------- 5. Overlay Aktual vs 3 Model (1 node representatif, di dalam sample_nodes) ----------
overlay_node = sample_nodes[0]
overlay_actual = all_targets_kph[:, overlay_node]
overlay_gnn = all_preds_kph[:, overlay_node]
overlay_lstm = all_preds_kph_lstm[:, overlay_node]
# ARIMA: ambil prediksi node ini dari arima_preds_by_node (sudah dihitung di CELL 45)
overlay_arima = arima_preds_by_node.get(overlay_node, np.full_like(overlay_actual, np.nan))

# ---------- 6. Residual (untuk Tab Analisis Error) ----------
residuals = {
    'gnn_ode': (all_preds_kph - all_targets_kph).flatten(),
    'lstm': (all_preds_kph_lstm - all_targets_kph_lstm).flatten(),
    'arima': arima_all_errors,
}
# actual value pasangan (utk cek bias overestimate/underestimate vs level kecepatan)
residual_actual_pairs = {
    'gnn_ode': all_targets_kph.flatten(),
    'lstm': all_targets_kph_lstm.flatten(),
    'arima': np.concatenate([arima_targets_by_node[n] for n in arima_preds_by_node]),
}

# ---------- 7. Bundling semua artefak non-model ke satu pickle ----------
bundle = {
    'node_lat': node_lat,
    'node_lon': node_lon,
    'edge_list_idx': edge_list_idx,
    'config': {**config, 'device': 'cpu'},
    'train_mean': float(train_mean),
    'train_std': float(train_std),
    'examples': examples,
    'metrics_table': {
        'full': comparison_full.to_dict(orient='records'),
        'sample': comparison_sample.to_dict(orient='records'),
        'sample_node_idx': sample_nodes,
    },
    'horizon_analysis': {
        'horizons_min': horizon_labels_min,
        'gnn_ode_mae': gnn_ode_mae_h, 'gnn_ode_rmse': gnn_ode_rmse_h,
        'lstm_mae': lstm_mae_h, 'lstm_rmse': lstm_rmse_h,
        'arima_mae': arima_mae_h, 'arima_rmse': arima_rmse_h,
    },
    'residuals': residuals,
    'residual_actual_pairs': residual_actual_pairs,
    'overlay': {
        'node_idx': overlay_node,
        'timestamps': [str(t) for t in test_timestamps],
        'actual': overlay_actual,
        'gnn_ode': overlay_gnn,
        'lstm': overlay_lstm,
        'arima': overlay_arima,
    },
    'global_rmse_note': {
        'gnn_ode_rmse_1step': float(rmse),
        'gnn_ode_mae_1step': float(mae),
        'test_span_days': 1,
    },
}

with open(os.path.join(ART_DIR, 'bundle.pkl'), 'wb') as f:
    pickle.dump(bundle, f)

# ---------- 8. Salin bobot model & edge_index (dibutuhkan utk inferensi LIVE di app.py) ----------
import shutil
shutil.copy('best_gnn_ode_model.pt', os.path.join(ART_DIR, 'best_gnn_ode_model.pt'))
shutil.copy('best_lstm_model.pt', os.path.join(ART_DIR, 'best_lstm_model.pt'))
torch.save(edge_index.cpu(), os.path.join(ART_DIR, 'edge_index.pt'))

print(f"\nSemua artefak tersimpan di folder '{ART_DIR}/':")
for fname in os.listdir(ART_DIR):
    path = os.path.join(ART_DIR, fname)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  - {fname}: {size_mb:.2f} MB")

print("\nLangkah selanjutnya: download/zip folder ini, taruh satu direktori dengan app.py,")
print("lalu jalankan `python app.py` (lihat requirements.txt untuk dependency).")


Skenario contoh berhasil diambil:
  - Jam Sibuk Pagi (Senin 08:00): shape (12, 2669), rata-rata 32.2 km/h
  - Jam Sibuk Sore (Senin 17:30): shape (12, 2669), rata-rata 29.1 km/h
  - Malam Lengang (Senin 21:00): shape (12, 2669), rata-rata 37.6 km/h
  - Weekend Siang (Sabtu 14:00): shape (12, 2669), rata-rata 30.5 km/h

=== Error vs Horizon (MAE, km/h) -- 20 node sample, konsisten dgn benchmarking ARIMA ===
 Horizon (menit)  GNN-ODE     LSTM    ARIMA
               5 0.600832 4.043638 0.751664
              10 1.057313 4.096479 0.821636
              15 1.609528 4.139667 0.958354

Semua artefak tersimpan di folder 'gradio_artifacts/':
  - bundle.pkl: 23.13 MB
  - best_gnn_ode_model.pt: 0.01 MB
  - best_lstm_model.pt: 3.34 MB
  - edge_index.pt: 0.09 MB

Langkah selanjutnya: download/zip folder ini, taruh satu direktori dengan app.py,
lalu jalankan `python app.py` (lihat requirements.txt untuk dependency).


In [ ]:
# ===== FLOW 5 (b) — Zip artefak untuk didownload dari Colab =====
import shutil
shutil.make_archive('gradio_artifacts', 'zip', 'gradio_artifacts')
print("Siap didownload: gradio_artifacts.zip")

from google.colab import files
files.download('gradio_artifacts.zip')


Siap didownload: gradio_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>